# Bài 2 trang 18: Phân tích và gom cụm khách hàng Mall

**Họ tên:** Lê Nguyễn Anh Du  
**MSSV:** 6451071011

Nguồn dữ liệu: [Mall_Customers.csv trên Google Drive](https://drive.google.com/file/d/1bsPcz8nZOMPcF3eVVs-EfmDRsqfjjUcC/view?usp=drive_link). Notebook thử tải từ Drive và có bản dữ liệu đã kiểm tra được nhúng sẵn để vẫn chạy khi liên kết không cho tải trực tiếp. Thực hiện đầy đủ các yêu cầu 1–9.

In [ ]:
print("Họ tên: Lê Nguyễn Anh Du | MSSV: 6451071011")
%pip -q install gdown plotly

In [ ]:
print("Họ tên: Lê Nguyễn Anh Du | MSSV: 6451071011")
import base64
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import gdown
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

HO_TEN = "Lê Nguyễn Anh Du"
MSSV = "6451071011"
def in_thong_tin():
    print(f"Họ tên: {HO_TEN} | MSSV: {MSSV}")

in_thong_tin()
print("Đã nhập các thư viện cần thiết.")

## 1) Nhập dữ liệu, thống kê thông tin cơ bản và xử lý dữ liệu thiếu

Các cột của tệp nguồn là `CustomerID`, `Gender`, `Age`, `Annual Income (k$)` và `Spending Score (1-100)`. Nếu có giá trị thiếu ở cột số, thay bằng trung vị; nếu thiếu giới tính, thay bằng giá trị xuất hiện nhiều nhất.

In [ ]:
in_thong_tin()
DRIVE_FILE_ID = "1bsPcz8nZOMPcF3eVVs-EfmDRsqfjjUcC"
CSV_BASE64 = "Q3VzdG9tZXJJRCxHZW5kZXIsQWdlLEFubnVhbCBJbmNvbWUgKGskKSxTcGVuZGluZyBTY29yZSAoMS0xMDApCjEsTWFsZSwxOSwxNSwzOQoyLE1hbGUsMjEsMTUsODEKMyxGZW1hbGUsMjAsMTYsNgo0LEZlbWFsZSwyMywxNiw3Nwo1LEZlbWFsZSwzMSwxNyw0MAo2LEZlbWFsZSwyMiwxNyw3Ngo3LEZlbWFsZSwzNSwxOCw2CjgsRmVtYWxlLDIzLDE4LDk0CjksTWFsZSw2NCwxOSwzCjEwLEZlbWFsZSwzMCwxOSw3MgoxMSxNYWxlLDY3LDE5LDE0CjEyLEZlbWFsZSwzNSwxOSw5OQoxMyxGZW1hbGUsNTgsMjAsMTUKMTQsRmVtYWxlLDI0LDIwLDc3CjE1LE1hbGUsMzcsMjAsMTMKMTYsTWFsZSwyMiwyMCw3OQoxNyxGZW1hbGUsMzUsMjEsMzUKMTgsTWFsZSwyMCwyMSw2NgoxOSxNYWxlLDUyLDIzLDI5CjIwLEZlbWFsZSwzNSwyMyw5OAoyMSxNYWxlLDM1LDI0LDM1CjIyLE1hbGUsMjUsMjQsNzMKMjMsRmVtYWxlLDQ2LDI1LDUKMjQsTWFsZSwzMSwyNSw3MwoyNSxGZW1hbGUsNTQsMjgsMTQKMjYsTWFsZSwyOSwyOCw4MgoyNyxGZW1hbGUsNDUsMjgsMzIKMjgsTWFsZSwzNSwyOCw2MQoyOSxGZW1hbGUsNDAsMjksMzEKMzAsRmVtYWxlLDIzLDI5LDg3CjMxLE1hbGUsNjAsMzAsNAozMixGZW1hbGUsMjEsMzAsNzMKMzMsTWFsZSw1MywzMyw0CjM0LE1hbGUsMTgsMzMsOTIKMzUsRmVtYWxlLDQ5LDMzLDE0CjM2LEZlbWFsZSwyMSwzMyw4MQozNyxGZW1hbGUsNDIsMzQsMTcKMzgsRmVtYWxlLDMwLDM0LDczCjM5LEZlbWFsZSwzNiwzNywyNgo0MCxGZW1hbGUsMjAsMzcsNzUKNDEsRmVtYWxlLDY1LDM4LDM1CjQyLE1hbGUsMjQsMzgsOTIKNDMsTWFsZSw0OCwzOSwzNgo0NCxGZW1hbGUsMzEsMzksNjEKNDUsRmVtYWxlLDQ5LDM5LDI4CjQ2LEZlbWFsZSwyNCwzOSw2NQo0NyxGZW1hbGUsNTAsNDAsNTUKNDgsRmVtYWxlLDI3LDQwLDQ3CjQ5LEZlbWFsZSwyOSw0MCw0Mgo1MCxGZW1hbGUsMzEsNDAsNDIKNTEsRmVtYWxlLDQ5LDQyLDUyCjUyLE1hbGUsMzMsNDIsNjAKNTMsRmVtYWxlLDMxLDQzLDU0CjU0LE1hbGUsNTksNDMsNjAKNTUsRmVtYWxlLDUwLDQzLDQ1CjU2LE1hbGUsNDcsNDMsNDEKNTcsRmVtYWxlLDUxLDQ0LDUwCjU4LE1hbGUsNjksNDQsNDYKNTksRmVtYWxlLDI3LDQ2LDUxCjYwLE1hbGUsNTMsNDYsNDYKNjEsTWFsZSw3MCw0Niw1Ngo2MixNYWxlLDE5LDQ2LDU1CjYzLEZlbWFsZSw2Nyw0Nyw1Mgo2NCxGZW1hbGUsNTQsNDcsNTkKNjUsTWFsZSw2Myw0OCw1MQo2NixNYWxlLDE4LDQ4LDU5CjY3LEZlbWFsZSw0Myw0OCw1MAo2OCxGZW1hbGUsNjgsNDgsNDgKNjksTWFsZSwxOSw0OCw1OQo3MCxGZW1hbGUsMzIsNDgsNDcKNzEsTWFsZSw3MCw0OSw1NQo3MixGZW1hbGUsNDcsNDksNDIKNzMsRmVtYWxlLDYwLDUwLDQ5Cjc0LEZlbWFsZSw2MCw1MCw1Ngo3NSxNYWxlLDU5LDU0LDQ3Cjc2LE1hbGUsMjYsNTQsNTQKNzcsRmVtYWxlLDQ1LDU0LDUzCjc4LE1hbGUsNDAsNTQsNDgKNzksRmVtYWxlLDIzLDU0LDUyCjgwLEZlbWFsZSw0OSw1NCw0Mgo4MSxNYWxlLDU3LDU0LDUxCjgyLE1hbGUsMzgsNTQsNTUKODMsTWFsZSw2Nyw1NCw0MQo4NCxGZW1hbGUsNDYsNTQsNDQKODUsRmVtYWxlLDIxLDU0LDU3Cjg2LE1hbGUsNDgsNTQsNDYKODcsRmVtYWxlLDU1LDU3LDU4Cjg4LEZlbWFsZSwyMiw1Nyw1NQo4OSxGZW1hbGUsMzQsNTgsNjAKOTAsRmVtYWxlLDUwLDU4LDQ2CjkxLEZlbWFsZSw2OCw1OSw1NQo5MixNYWxlLDE4LDU5LDQxCjkzLE1hbGUsNDgsNjAsNDkKOTQsRmVtYWxlLDQwLDYwLDQwCjk1LEZlbWFsZSwzMiw2MCw0Mgo5NixNYWxlLDI0LDYwLDUyCjk3LEZlbWFsZSw0Nyw2MCw0Nwo5OCxGZW1hbGUsMjcsNjAsNTAKOTksTWFsZSw0OCw2MSw0MgoxMDAsTWFsZSwyMCw2MSw0OQoxMDEsRmVtYWxlLDIzLDYyLDQxCjEwMixGZW1hbGUsNDksNjIsNDgKMTAzLE1hbGUsNjcsNjIsNTkKMTA0LE1hbGUsMjYsNjIsNTUKMTA1LE1hbGUsNDksNjIsNTYKMTA2LEZlbWFsZSwyMSw2Miw0MgoxMDcsRmVtYWxlLDY2LDYzLDUwCjEwOCxNYWxlLDU0LDYzLDQ2CjEwOSxNYWxlLDY4LDYzLDQzCjExMCxNYWxlLDY2LDYzLDQ4CjExMSxNYWxlLDY1LDYzLDUyCjExMixGZW1hbGUsMTksNjMsNTQKMTEzLEZlbWFsZSwzOCw2NCw0MgoxMTQsTWFsZSwxOSw2NCw0NgoxMTUsRmVtYWxlLDE4LDY1LDQ4CjExNixGZW1hbGUsMTksNjUsNTAKMTE3LEZlbWFsZSw2Myw2NSw0MwoxMTgsRmVtYWxlLDQ5LDY1LDU5CjExOSxGZW1hbGUsNTEsNjcsNDMKMTIwLEZlbWFsZSw1MCw2Nyw1NwoxMjEsTWFsZSwyNyw2Nyw1NgoxMjIsRmVtYWxlLDM4LDY3LDQwCjEyMyxGZW1hbGUsNDAsNjksNTgKMTI0LE1hbGUsMzksNjksOTEKMTI1LEZlbWFsZSwyMyw3MCwyOQoxMjYsRmVtYWxlLDMxLDcwLDc3CjEyNyxNYWxlLDQzLDcxLDM1CjEyOCxNYWxlLDQwLDcxLDk1CjEyOSxNYWxlLDU5LDcxLDExCjEzMCxNYWxlLDM4LDcxLDc1CjEzMSxNYWxlLDQ3LDcxLDkKMTMyLE1hbGUsMzksNzEsNzUKMTMzLEZlbWFsZSwyNSw3MiwzNAoxMzQsRmVtYWxlLDMxLDcyLDcxCjEzNSxNYWxlLDIwLDczLDUKMTM2LEZlbWFsZSwyOSw3Myw4OAoxMzcsRmVtYWxlLDQ0LDczLDcKMTM4LE1hbGUsMzIsNzMsNzMKMTM5LE1hbGUsMTksNzQsMTAKMTQwLEZlbWFsZSwzNSw3NCw3MgoxNDEsRmVtYWxlLDU3LDc1LDUKMTQyLE1hbGUsMzIsNzUsOTMKMTQzLEZlbWFsZSwyOCw3Niw0MAoxNDQsRmVtYWxlLDMyLDc2LDg3CjE0NSxNYWxlLDI1LDc3LDEyCjE0NixNYWxlLDI4LDc3LDk3CjE0NyxNYWxlLDQ4LDc3LDM2CjE0OCxGZW1hbGUsMzIsNzcsNzQKMTQ5LEZlbWFsZSwzNCw3OCwyMgoxNTAsTWFsZSwzNCw3OCw5MAoxNTEsTWFsZSw0Myw3OCwxNwoxNTIsTWFsZSwzOSw3OCw4OAoxNTMsRmVtYWxlLDQ0LDc4LDIwCjE1NCxGZW1hbGUsMzgsNzgsNzYKMTU1LEZlbWFsZSw0Nyw3OCwxNgoxNTYsRmVtYWxlLDI3LDc4LDg5CjE1NyxNYWxlLDM3LDc4LDEKMTU4LEZlbWFsZSwzMCw3OCw3OAoxNTksTWFsZSwzNCw3OCwxCjE2MCxGZW1hbGUsMzAsNzgsNzMKMTYxLEZlbWFsZSw1Niw3OSwzNQoxNjIsRmVtYWxlLDI5LDc5LDgzCjE2MyxNYWxlLDE5LDgxLDUKMTY0LEZlbWFsZSwzMSw4MSw5MwoxNjUsTWFsZSw1MCw4NSwyNgoxNjYsRmVtYWxlLDM2LDg1LDc1CjE2NyxNYWxlLDQyLDg2LDIwCjE2OCxGZW1hbGUsMzMsODYsOTUKMTY5LEZlbWFsZSwzNiw4NywyNwoxNzAsTWFsZSwzMiw4Nyw2MwoxNzEsTWFsZSw0MCw4NywxMwoxNzIsTWFsZSwyOCw4Nyw3NQoxNzMsTWFsZSwzNiw4NywxMAoxNzQsTWFsZSwzNiw4Nyw5MgoxNzUsRmVtYWxlLDUyLDg4LDEzCjE3NixGZW1hbGUsMzAsODgsODYKMTc3LE1hbGUsNTgsODgsMTUKMTc4LE1hbGUsMjcsODgsNjkKMTc5LE1hbGUsNTksOTMsMTQKMTgwLE1hbGUsMzUsOTMsOTAKMTgxLEZlbWFsZSwzNyw5NywzMgoxODIsRmVtYWxlLDMyLDk3LDg2CjE4MyxNYWxlLDQ2LDk4LDE1CjE4NCxGZW1hbGUsMjksOTgsODgKMTg1LEZlbWFsZSw0MSw5OSwzOQoxODYsTWFsZSwzMCw5OSw5NwoxODcsRmVtYWxlLDU0LDEwMSwyNAoxODgsTWFsZSwyOCwxMDEsNjgKMTg5LEZlbWFsZSw0MSwxMDMsMTcKMTkwLEZlbWFsZSwzNiwxMDMsODUKMTkxLEZlbWFsZSwzNCwxMDMsMjMKMTkyLEZlbWFsZSwzMiwxMDMsNjkKMTkzLE1hbGUsMzMsMTEzLDgKMTk0LEZlbWFsZSwzOCwxMTMsOTEKMTk1LEZlbWFsZSw0NywxMjAsMTYKMTk2LEZlbWFsZSwzNSwxMjAsNzkKMTk3LEZlbWFsZSw0NSwxMjYsMjgKMTk4LE1hbGUsMzIsMTI2LDc0CjE5OSxNYWxlLDMyLDEzNywxOAoyMDAsTWFsZSwzMCwxMzcsODM="
required_columns = {"CustomerID", "Gender", "Age", "Annual Income (k$)", "Spending Score (1-100)"}
try:
    downloaded_file = gdown.download(id=DRIVE_FILE_ID, output="Mall_Customers.csv", quiet=True)
    if downloaded_file is None:
        raise RuntimeError("Không tải được tệp từ Drive")
    df = pd.read_csv(downloaded_file)
    if not required_columns.issubset(df.columns):
        raise ValueError("Tệp tải về không có đủ cột dữ liệu")
    print("Đã đọc dữ liệu trực tiếp từ Google Drive.")
except Exception as exc:
    df = pd.read_csv(BytesIO(base64.b64decode(CSV_BASE64)))
    print(f"Đã đọc bản dữ liệu nhúng từ cùng tệp nguồn ({type(exc).__name__}).")

print("Kích thước dữ liệu:", df.shape)
print("5 dòng đầu tiên:")
display(df.head())
print("Thông tin cột:")
df.info()
print("Thống kê mô tả:")
display(df.describe(include="all").transpose())

In [ ]:
in_thong_tin()
print("Số giá trị thiếu trước khi xử lý:")
print(df.isna().sum().to_string())

# Làm sạch các cột và chuyển cột số về dạng số học.
df["Gender"] = df["Gender"].astype("string").str.strip().str.title()
df["Gender"] = df["Gender"].replace("", pd.NA)
numeric_columns = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")
    df[column] = df[column].fillna(df[column].median())
if df["Gender"].isna().any():
    df["Gender"] = df["Gender"].fillna(df["Gender"].mode().iloc[0])

print("Số giá trị thiếu sau khi xử lý:")
print(df.isna().sum().to_string())
print("Kích thước dữ liệu sau xử lý:", df.shape)

## 2) Biểu đồ Histogram của Tuổi, Thu nhập hằng năm và Điểm mua sắm

In [ ]:
in_thong_tin()
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
chart_info = [
    ("Age", "Tuổi", "tab:blue"),
    ("Annual Income (k$)", "Thu nhập hằng năm (nghìn USD)", "tab:green"),
    ("Spending Score (1-100)", "Điểm mua sắm", "tab:orange")
]
for ax, (column, label, color) in zip(axes, chart_info):
    ax.hist(df[column], bins=15, color=color, edgecolor="white")
    ax.set_title(f"Phân phối {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("Tần suất")
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 3) Thống kê số mẫu theo Giới tính

In [ ]:
in_thong_tin()
gender_counts = df["Gender"].value_counts()
print(gender_counts.to_string())
plt.figure(figsize=(6, 4))
plt.bar(gender_counts.index.astype(str), gender_counts.values, color=["#e76f51", "#457b9d"])
plt.title("Số khách hàng theo giới tính")
plt.xlabel("Giới tính")
plt.ylabel("Số mẫu")
for i, count in enumerate(gender_counts.values):
    plt.text(i, count + 1, str(count), ha="center")
plt.tight_layout()
plt.show()

## 4) Scatter plot: Tuổi và Thu nhập hằng năm theo Giới tính

In [ ]:
in_thong_tin()
plt.figure(figsize=(8, 5))
for gender, group in df.groupby("Gender"):
    plt.scatter(group["Age"], group["Annual Income (k$)"], label=gender, alpha=0.7)
plt.title("Tuổi và Thu nhập hằng năm theo giới tính")
plt.xlabel("Tuổi")
plt.ylabel("Thu nhập hằng năm (nghìn USD)")
plt.legend(title="Giới tính")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5) Scatter plot: Thu nhập hằng năm và Điểm mua sắm theo Giới tính

In [ ]:
in_thong_tin()
plt.figure(figsize=(8, 5))
for gender, group in df.groupby("Gender"):
    plt.scatter(group["Annual Income (k$)"], group["Spending Score (1-100)"], label=gender, alpha=0.7)
plt.title("Thu nhập hằng năm và Điểm mua sắm theo giới tính")
plt.xlabel("Thu nhập hằng năm (nghìn USD)")
plt.ylabel("Điểm mua sắm")
plt.legend(title="Giới tính")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Các hàm dùng chung cho phương pháp khuỷu tay và k-Means

Với mỗi nhóm thuộc tính, thử `k` từ 1 đến 10 và tính Inertia (WSS). Hàm gợi ý điểm khuỷu tay bằng khoảng cách lớn nhất từ đường Inertia đã chuẩn hóa đến đoạn thẳng nối hai đầu đường cong. Luôn xem biểu đồ để diễn giải lựa chọn `k`.

In [ ]:
def elbow_analysis(X, title, k_max=10, selected_k=None):
    k_values = np.arange(1, min(k_max, len(X)) + 1)
    inertias = []
    for k in k_values:
        km = KMeans(n_clusters=int(k), init="k-means++", n_init=10, random_state=42)
        km.fit(X)
        inertias.append(km.inertia_)
    inertias = np.asarray(inertias)

    # Chuẩn hóa hai trục về [0, 1], lấy điểm xa đường nối hai đầu nhất.
    x_norm = (k_values - k_values[0]) / (k_values[-1] - k_values[0])
    y_norm = (inertias - inertias[-1]) / (inertias[0] - inertias[-1])
    distances = np.abs(y_norm - (1 - x_norm)) / np.sqrt(2)
    suggested_k = int(k_values[1:-1][np.argmax(distances[1:-1])])

    in_thong_tin()
    print(f"{title}: k được gợi ý từ điểm khuỷu tay = {suggested_k}")
    plt.figure(figsize=(7, 4))
    plt.plot(k_values, inertias, marker="o")
    plot_k = selected_k if selected_k is not None else suggested_k
    plt.axvline(plot_k, color="tab:red", linestyle="--", label=f"k được chọn={plot_k}")
    plt.title(f"Phương pháp khuỷu tay — {title}")
    plt.xlabel("Số cụm k")
    plt.ylabel("Inertia (WSS)")
    plt.xticks(k_values)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
    return suggested_k, inertias

def fit_clusters(X, k):
    model = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = model.fit_predict(X)
    score = silhouette_score(X, labels)
    return model, labels, score

in_thong_tin()
print("Đã định nghĩa các hàm tính Inertia, gợi ý điểm khuỷu tay và gom cụm.")

## 6–8) Gom cụm theo Tuổi và Điểm mua sắm

Theo hướng dẫn của bài, chọn **k = 4** sau khi xem biểu đồ khuỷu tay. Biểu diễn các cụm và tâm cụm, rồi tính Silhouette coefficient.

In [ ]:
in_thong_tin()
age_score_features = ["Age", "Spending Score (1-100)"]
X_age_score = df[age_score_features].to_numpy()
suggested_k_age_score, inertia_age_score = elbow_analysis(X_age_score, "Tuổi và Điểm mua sắm", selected_k=4)
k_age_score = 4
age_score_model, age_score_labels, age_score_silhouette = fit_clusters(X_age_score, k_age_score)
print(f"Số cụm được chọn theo hướng dẫn: k={k_age_score}")

In [ ]:
in_thong_tin()
plt.figure(figsize=(8, 5))
for cluster_id in range(k_age_score):
    points = X_age_score[age_score_labels == cluster_id]
    plt.scatter(points[:, 0], points[:, 1], label=f"Cụm {cluster_id + 1}", alpha=0.75)
plt.scatter(age_score_model.cluster_centers_[:, 0], age_score_model.cluster_centers_[:, 1],
            marker="X", s=220, c="black", label="Tâm cụm")
plt.title("Gom cụm theo Tuổi và Điểm mua sắm")
plt.xlabel("Tuổi")
plt.ylabel("Điểm mua sắm")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
in_thong_tin()
print(f"Silhouette coefficient — Tuổi và Điểm mua sắm: {age_score_silhouette:.4f}")

## 9a) Gom cụm theo Thu nhập hằng năm và Điểm mua sắm

Dùng lại phương pháp khuỷu tay để chọn `k`, vẽ các cụm và tính Silhouette coefficient.

In [ ]:
in_thong_tin()
income_score_features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_income_score = df[income_score_features].to_numpy()
k_income_score, inertia_income_score = elbow_analysis(X_income_score, "Thu nhập và Điểm mua sắm")
income_score_model, income_score_labels, income_score_silhouette = fit_clusters(X_income_score, k_income_score)
print(f"Đã gom cụm Thu nhập và Điểm mua sắm với k={k_income_score}.")

In [ ]:
in_thong_tin()
plt.figure(figsize=(8, 5))
for cluster_id in range(k_income_score):
    points = X_income_score[income_score_labels == cluster_id]
    plt.scatter(points[:, 0], points[:, 1], label=f"Cụm {cluster_id + 1}", alpha=0.75)
plt.scatter(income_score_model.cluster_centers_[:, 0], income_score_model.cluster_centers_[:, 1],
            marker="X", s=220, c="black", label="Tâm cụm")
plt.title("Gom cụm theo Thu nhập và Điểm mua sắm")
plt.xlabel("Thu nhập hằng năm (nghìn USD)")
plt.ylabel("Điểm mua sắm")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
in_thong_tin()
print(f"Silhouette coefficient — Thu nhập và Điểm mua sắm: {income_score_silhouette:.4f}")

## 9b) Gom cụm theo Tuổi, Thu nhập hằng năm và Điểm mua sắm

Áp dụng Elbow, k-Means và Silhouette coefficient như trên. Dùng Plotly để biểu diễn các cụm trong không gian 3D.

In [ ]:
in_thong_tin()
features_3d = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
X_3d = df[features_3d].to_numpy()
k_3d, inertia_3d = elbow_analysis(X_3d, "Tuổi, Thu nhập và Điểm mua sắm")
model_3d, labels_3d, silhouette_3d = fit_clusters(X_3d, k_3d)
print(f"Đã gom cụm ba thuộc tính với k={k_3d}.")

In [ ]:
in_thong_tin()
plot_df = df[features_3d].copy()
plot_df["Cụm"] = [f"Cụm {cluster_id + 1}" for cluster_id in labels_3d]
fig = px.scatter_3d(
    plot_df,
    x="Age", y="Annual Income (k$)", z="Spending Score (1-100)",
    color="Cụm",
    title="Gom cụm khách hàng theo ba thuộc tính",
    labels={"Age": "Tuổi", "Annual Income (k$)": "Thu nhập hằng năm (nghìn USD)",
            "Spending Score (1-100)": "Điểm mua sắm"},
    opacity=0.8
)
fig.add_scatter3d(
    x=model_3d.cluster_centers_[:, 0],
    y=model_3d.cluster_centers_[:, 1],
    z=model_3d.cluster_centers_[:, 2],
    mode="markers", name="Tâm cụm",
    marker=dict(size=7, color="black", symbol="diamond")
)
fig.update_layout(height=650)
fig.show()

In [ ]:
in_thong_tin()
print(f"Silhouette coefficient — ba thuộc tính: {silhouette_3d:.4f}")
print("Silhouette coefficient càng gần 1 thì các cụm càng tách biệt và chặt chẽ.")